In [ ]:
# Keras machine learning for T1D

In [ ]:
#import pyreadr
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import itertools as it
import pandas as pd
from pandas.plotting import scatter_matrix
import scipy
import numpy as np
import xgboost
from catboost import CatBoostClassifier

In [ ]:
from sklearn.preprocessing import LabelBinarizer
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import numpy as np
import argparse

In [ ]:
import sklearn
import seaborn as sns
import scipy
from sklearn import svm
from sklearn.metrics import RocCurveDisplay, auc
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc, RocCurveDisplay


In [ ]:
from scipy.stats import mannwhitneyu
from scipy import stats
from sklearn import metrics
import statsmodels.formula.api as smf
import statsmodels.api as sm

In [ ]:
from pandas import read_csv
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn import preprocessing
import warnings
warnings.filterwarnings('ignore')

In [ ]:
T1df=pd.read_csv("/cellar/users/tsears/projects/T1D/ALL5_199_TOPMED_SUSIE_HLA_T1D_signals_updateID.vcf",sep="\t",skiprows=27)

In [ ]:
T1df.index=T1df["ID"]
T1df=T1df.drop(["#CHROM","POS","ID","REF","ALT","QUAL","FILTER","INFO","FORMAT"],axis=1)
T1df.columns=[x.split("_",2)[2] for x in T1df.columns]
T1df=np.transpose(T1df)
T1df

In [ ]:
# read in metadata
meta=pd.read_csv("/cellar/users/tsears/projects/T1D/meta_all5_cat_cov.ped",sep=" ")
meta

In [ ]:
# merge datasets
meta_T1df=pd.merge(T1df,meta,left_index=True,right_on="#FID")
meta_T1df.index=meta_T1df["#FID"]
meta_T1df["GENIEUK"]=meta_T1df["GENIE_UK"]
meta_T1df=meta_T1df.drop(["#FID","IID","GENIE_UK"],axis=1)

In [ ]:
#Map 0/0 stuff to snps 
cols2map=[x for x in list(meta_T1df.columns) if 'chr' in x]
cols2map2=[x for x in list(meta_T1df.columns) if 'rs' in x]
cols2map3=[x for x in list(meta_T1df.columns) if '_' in x]
#cols2map3=[x for x in list(meta_T1df.columns) if 'chr' in x]
cols2map_all=cols2map+cols2map2+cols2map3

for col in cols2map_all:
    meta_T1df[col]=meta_T1df[col].map({"0/0":0,"0/1":1,"1/0":1,"1/1":2})
    
meta_T1df

In [ ]:
meta_T1df.to_csv('/cellar/users/tsears/projects/T1D/formattedVCF.txt',sep='\t')

In [ ]:
meta_T1df['GENIEUK'].value_counts()

In [ ]:
meta_T1df_dropped=meta_T1df#[meta_T1df["GENIEUK"]==0]
meta_T1df_dropped.shape[0]

meta_T1df_dropped_nonHLA=meta_T1df_dropped.filter(regex="chr")
meta_T1df_dropped_HLA=meta_T1df_dropped.loc[:,list(~meta_T1df_dropped.columns.isin(meta_T1df_dropped_nonHLA.columns))]

meta_T1df_dropped_HLA=meta_T1df_dropped_HLA.drop(['SEX',
 'T1DGC',
 'DCCT',
 'GoKIND',"PC1","PC2","PC3","PC4",
 'GENIEUK'],axis=1)

meta_T1df_dropped=meta_T1df_dropped.drop(['SEX',
 'T1DGC',
 'DCCT',
 'GoKIND',"PC1","PC2","PC3","PC4",
 'GENIEUK'],axis=1)

#meta_T1df_dropped.iloc[:5,:].to_csv("/cellar/users/tsears/projects/T1D/SampleVCFs/ALL.txt",sep="\t")
#meta_T1df_dropped_HLA.iloc[:5,:].to_csv("/cellar/users/tsears/projects/T1D/SampleVCFs/HLA.txt",sep="\t")
#meta_T1df_dropped_nonHLA.iloc[:5,:].to_csv("/cellar/users/tsears/projects/T1D/SampleVCFs/nonHLA.txt",sep="\t")

testing_df=meta_T1df_dropped["DISEASE"]
training_df=meta_T1df_dropped.drop(["DISEASE"],axis=1)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, RocCurveDisplay
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier

# Cross-validation setup
n_splits = 10
random_state = 217
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

# Initialize CatBoost classifier with similar hyperparameters
classifier = CatBoostClassifier(
    iterations=254,  # Equivalent to n_estimators
    depth=5,  # Equivalent to max_depth
    learning_rate=0.12, 
    l2_leaf_reg=3.05,  # Equivalent to reg_lambda
    #boosting_type='Ordered',  # More similar to Dart boosting in XGBoost
    subsample=0.725,  # Keep subsample same as XGBoost
    border_count=80,  # Similar to XGBoost 'hist' method
    scale_pos_weight=4,  # Same class balancing as XGBoost
    random_seed=random_state,
    bagging_temperature=2,
    random_strength=2.0,
    colsample_bylevel=0.9,
    leaf_estimation_method='Gradient',
    verbose=False,  # Silence training logs
    grow_policy='Lossguide'
)

# Lists to store results
tprs = []
aucs = []
mean_fpr = np.linspace(0, 1, 100)
probas_list = []  # Store the probabilities for each cv
IDs = []

fig, ax = plt.subplots(figsize=(6, 6))

# Cross-validation loop
for fold, (train, test) in enumerate(cv.split(X, Y)):
    
    #if fold != 0:
    #    continue  # Only run fold 0 as per original script

    classifier.fit(X[train], Y[train])
    probabilities = classifier.predict_proba(X[test])[:, 1]  # Get probabilities for positive class
    probas_list.append(probabilities)
    IDs.append(meta_T1df_dropped.index[test])
    
    # Compute ROC curve and ROC area
    fpr, tpr, thresholds = roc_curve(Y[test], probabilities)
    viz = RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=auc(fpr, tpr), estimator_name=f"ROC fold {fold}")
    viz.plot(ax=ax, alpha=0.3, lw=1, name=f"ROC fold {fold}")
    
    interp_tpr = np.interp(mean_fpr, fpr, tpr)
    interp_tpr[0] = 0.0
    tprs.append(interp_tpr)
    aucs.append(viz.roc_auc)
    print(f"Fold {fold} AUC: {viz.roc_auc}")
    
# Compute mean ROC curve
mean_tpr = np.mean(tprs, axis=0)
mean_tpr[-1] = 1.0
mean_auc = auc(mean_fpr, mean_tpr)
std_auc = np.std(aucs)

ax.plot(
    mean_fpr,
    mean_tpr,
    color="b",
    label=r"Mean ROC (AUC = %0.2f ± %0.2f)" % (mean_auc, std_auc),
    lw=2,
    alpha=0.8,
)

# Plot standard deviation shading
std_tpr = np.std(tprs, axis=0)
tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
ax.fill_between(
    mean_fpr,
    tprs_lower,
    tprs_upper,
    color="grey",
    alpha=0.2,
    label=r"$\pm$ 1 std. dev.",
)

# Final plot settings
ax.set(
    xlabel="False Positive Rate",
    ylabel="True Positive Rate",
    title="Mean ROC curve with variability",
)
ax.legend(loc="lower right")
plt.show()

print(f"Mean AUC: {mean_auc} ± {std_auc}")


# Test on T2 diabetes patients

In [ ]:
#retrain final classifier on all samples
random_state=217

classifier_final = xgboost.XGBClassifier(n_estimators=910, random_state=random_state, learning_rate=0.8, max_depth=34, reg_alpha=0,
                                     booster='dart', rate_drop=0.2, subsample=0.6, reg_lambda=0, gamma=0,tree_method='hist')

classifier_final.fit(training_df, testing_df)

# Create XGBClassifier
#classifier_final = xgboost.XGBClassifier()

# Load model parameters from your existing booster
#classifier_final.load_model('/cellar/users/tsears/projects/T1D/XGBoostModels/ALL_NoPCs_199.ubj')



In [ ]:
classifier_final.save_model('/cellar/users/tsears/projects/T1D/XGBoostModels/ALL_NoPCs_2025.ubj')

In [ ]:
classifier_final.load_model('/cellar/users/tsears/projects/T1D/XGBoostModels/ALL_NoPCs_2025.ubj')


### AUC on T2 Discovery patients

In [ ]:
#quick test on T2 patients
T2=pd.read_csv("~/projects/T1D/T2_discovery_df.txt",sep="\t",index_col=0)
T2_test=T2.drop(['id1','id2.1','2','3','SEX','T1D'],axis=1)

T2_test['PC1']=np.nan
T2_test['PC2']=np.nan
T2_test['PC3']=np.nan
T2_test['PC4']=np.nan
T2_test['SEX']=np.nan
T2_test['GoKIND']=np.nan
T2_test['T1DGC']=np.nan
T2_test['GENIEUK']=np.nan
T2_test['DCCT']=np.nan


probas_T2=list(classifier_final.predict_proba(T2_test)[:, 1])

flat_ids = T2['id2.1']
flat_probs = probas_T2

# Create a DataFrame
df = pd.DataFrame({'CV_ID': flat_ids, 'Probability': flat_probs})

In [ ]:
#export df
#df.to_csv("/cellar/users/tsears/projects/T1D/tables/delong_ALL_T2D_discovery_noPCs.txt",sep="\t")

In [ ]:
fpr, tpr, threshold = metrics.roc_curve(T2["T1D"],probas_T2,pos_label=2)
roc_auc = metrics.auc(fpr, tpr)

# method I: plt
import matplotlib.pyplot as plt
plt.figure(figsize=(8,8))

plt.title('Receiver Operating Characteristic')
plt.plot(fpr, tpr, 'b', label = 'TJ AUC = %0.5f' % roc_auc)

plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.savefig("/cellar/users/tsears/projects/T1D/plots/ALL_T2D_discovery.pdf",bbox_inches = "tight")
plt.show()

### AUC on T2 Discovery patients

In [ ]:
#quick test on T2 patients
T2=pd.read_csv("~/projects/T1D/T2_test_df.txt",sep="\t",index_col=0)
T2_test=T2.drop(['id1','id2.1','2','3','SEX','T1D'],axis=1)

probas_T2=list(classifier_final.predict_proba(T2_test)[:, 1])

flat_ids = T2['id2.1']
flat_probs = probas_T2

# Create a DataFrame
df = pd.DataFrame({'CV_ID': flat_ids, 'Probability': flat_probs})

In [ ]:
#export df
df.to_csv("/cellar/users/tsears/projects/T1D/tables/delong_ALL_T2D_test_noPCs.txt",sep="\t")

In [ ]:
fpr, tpr, threshold = metrics.roc_curve(T2["T1D"],probas_T2,pos_label=2)
roc_auc = metrics.auc(fpr, tpr)

# method I: plt
import matplotlib.pyplot as plt
plt.figure(figsize=(8,8))

plt.title('Receiver Operating Characteristic')
plt.plot(fpr, tpr, 'b', label = 'TJ AUC = %0.5f' % roc_auc)

plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.savefig("/cellar/users/tsears/projects/T1D/plots/ALL_T2D_test.pdf",bbox_inches = "tight")
plt.show()

# Test on held-out NPOD cohort

In [ ]:
#quick test on T2 patients
T2=pd.read_csv("~/projects/T1D/NPOD_df.txt",sep="\t",index_col=0)
T2_test=T2.drop(['id1','id2.1','2','3','SEX','T1D'],axis=1)

#T2_test['PC1']=np.nan
#T2_test['PC2']=np.nan
#T2_test['PC3']=np.nan
#T2_test['PC4']=np.nan
#T2_test['SEX']=np.nan
#T2_test['GoKIND']=np.nan
#T2_test['T1DGC']=np.nan
#T2_test['GENIEUK']=np.nan
#T2_test['DCCT']=np.nan

probas_T2=list(classifier_final.predict_proba(T2_test)[:, 1])

flat_ids = T2['id2.1']
flat_probs = probas_T2

# Create a DataFrame
df = pd.DataFrame({'CV_ID': flat_ids, 'Probability': flat_probs})

In [ ]:
#export df
#df.to_csv("/cellar/users/tsears/projects/T1D/tables/delong_ALL_NPOD_test_noPCs.txt",sep="\t")

In [ ]:
fpr, tpr, threshold = metrics.roc_curve(T2["T1D"],probas_T2,pos_label=2)
roc_auc = metrics.auc(fpr, tpr)

# method I: plt
import matplotlib.pyplot as plt
plt.figure(figsize=(8,8))

plt.title('Receiver Operating Characteristic')
plt.plot(fpr, tpr, 'b', label = 'TJ AUC = %0.5f' % roc_auc)

plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.savefig("/cellar/users/tsears/projects/T1D/plots/ALL_NPOD_test.pdf",bbox_inches = "tight")
plt.show()
#8879

# Test on nonDR3DR4

In [ ]:
# load DR3DR4 data, and exclude them from new classifier
T2=pd.read_csv("~/projects/T1D/NonDR3DR4_df.txt",sep="\t",index_col=0)
full_df=T2
T2=T2.drop(['#FID','IID.1','SEX','DISEASE','PC1','PC2','PC3','PC4','T1DGC','DCCT','GENIE_UK','GoKIND'],axis=1)
T2=T2.loc[:,[x in training_df.columns for x in T2.columns]]
T2

In [ ]:
#retrain final classifier on all NON DRD3DR4 samples
classifier_final = xgboost.XGBClassifier(n_estimators=910, random_state=random_state, learning_rate=0.8, max_depth=34, reg_alpha=0,
                                     booster='dart', rate_drop=0.2, subsample=0.6, reg_lambda=0, gamma=0,tree_method='hist')

classifier_final.fit(training_df[[x not in T2.index for x in training_df.index]], testing_df[[x not in T2.index for x in training_df.index]])

In [ ]:
#quick test on T2 patients
probas_T2=list(classifier_final.predict_proba(T2)[:, 1])

flat_ids = T2.index
flat_probs = probas_T2

# Create a DataFrame
df = pd.DataFrame({'CV_ID': flat_ids, 'Probability': flat_probs})

In [ ]:
#export df
df.to_csv("/cellar/users/tsears/projects/T1D/tables/delong_ALL_nonDRD3DR4_noPCs.txt",sep="\t")


In [ ]:
fpr, tpr, threshold = metrics.roc_curve(full_df["DISEASE"],probas_T2,pos_label=2)
roc_auc = metrics.auc(fpr, tpr)

# method I: plt
import matplotlib.pyplot as plt
plt.figure(figsize=(8,8))

plt.title('Receiver Operating Characteristic')
plt.plot(fpr, tpr, 'b', label = 'TJ AUC = %0.5f' % roc_auc)

plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
#plt.savefig("/cellar/users/tsears/projects/T1D/plots/ALL_DR3DR4.pdf",bbox_inches = "tight")
plt.show()

### Delong test vs previous performance

In [ ]:
# load in dataset
# merge IDs and probabilities
flat_ids = [item for sublist in IDs for item in sublist]
flat_probs = [item for sublist in probas_list for item in sublist]

# Create a DataFrame
df = pd.DataFrame({'CV_ID': flat_ids, 'Probability': flat_probs})

GRS2=pd.read_csv("/cellar/users/tsears/projects/T1D/ALL5_merged_GRS2_remove_dup_alleles_new_beta_new_11_20_23.txt",sep="\t")
GRS2=GRS2[['FID','sum_All5_nonHLA','Total_sum_HLA','Total_sum','disease']]

# merge with results
delong_export=pd.merge(df,GRS2,left_on="CV_ID",right_on="FID",how="left")
delong_plot=pd.merge(df,GRS2,left_on="CV_ID",right_on="FID")

delong_export.to_csv("/cellar/users/tsears/projects/T1D/delong_ALL_V2_noPCs.txt",sep="\t")

In [ ]:
validation_plot=delong_plot

import sklearn.metrics as metrics
from scipy import stats
import scikit_posthocs as sp
#import pyroc

fpr, tpr, threshold = metrics.roc_curve(validation_plot["disease"],validation_plot["Probability"],pos_label=2)
roc_auc = metrics.auc(fpr, tpr)
fpr1, tpr1, threshold1 = metrics.roc_curve(validation_plot["disease"],validation_plot["Total_sum"],pos_label=2)
roc_auc1 = metrics.auc(fpr1, tpr1)

p_value=0.134#delong_roc_test(validation_plot['disease'], validation_plot['Probability'], validation_plot['sum_All5_nonHLA'])

# method I: plt
import matplotlib.pyplot as plt
plt.figure(figsize=(8,8))

plt.text(0.6, 0.2, f'DeLong Test p-value = {p_value:.3f}', fontsize=12)

plt.title('Receiver Operating Characteristic')
plt.plot(fpr, tpr, 'b', label = 'TJ AUC = %0.5f' % roc_auc)
plt.plot(fpr1, tpr1, 'r', label = 'GRS2 AUC = %0.5f' % roc_auc1)

plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],'r--')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.savefig("/cellar/users/tsears/projects/T1D/All_AUC_noPCs.pdf",bbox_inches = "tight")
plt.show()


# Feature importance and interaction

In [ ]:
shap.plots._waterfall.waterfall_legacy(explainer.expected_value,shap_values[1535], max_display=30,feature_names = X.columns,show=False)
plt.savefig("/cellar/users/tsears/projects/T1D/plots/All_WTCCC1-T1D-A_WTCCC168357_waterfall.pdf",bbox_inches='tight')
